# Vera su misura · addestramento

Il pacchetto lo prepara Prisma sul Mac: solo esempi resi anonimi e approvati da te. Questo notebook non contiene dati.

1. Scegli una GPU (Colab: Runtime › Cambia tipo di runtime › GPU T4; Kaggle: Accelerator › GPU).
2. Carica la cartella del pacchetto (o lo zip) qui accanto, con `addestramento.jsonl` e `manifest.json`.
3. Esegui le celle in ordine: circa due ore.
4. Scarica `vera-su-misura.zip` (il file .gguf e `rapporto.json`) e importalo in Prisma.

I valori di avvio (r, alpha, epoche, lr) sono quelli consigliati da Unsloth: da confermare alla prima prova.

In [ ]:
# 1. Unsloth (la versione si fissa nel README dopo la prima prova vera)
!pip install unsloth

In [ ]:
# 2. Il manifesto del pacchetto
import json, os, zipfile
if os.path.exists('pacchetto.zip'):
    zipfile.ZipFile('pacchetto.zip').extractall('.')
manifesto = json.load(open('manifest.json', encoding='utf-8'))
modello_partenza = manifesto['modelloUnsloth']  # da verificare su Hugging Face
lunghezza = int(manifesto.get('lunghezzaMassima', 4096))
print(manifesto['esempi'], 'esempi · da', modello_partenza, '·', manifesto['nomeModello'])

In [ ]:
# 3. Il modello di partenza in 4 bit
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(model_name=modello_partenza, max_seq_length=lunghezza, load_in_4bit=True)

In [ ]:
# 4. LoRA
model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=16, lora_dropout=0,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth')

In [ ]:
# 5. Gli esempi (addestramento.jsonl: una conversazione per riga)
from datasets import load_dataset
dati = load_dataset('json', data_files='addestramento.jsonl', split='train')
dati = dati.map(lambda e: {'text': tokenizer.apply_chat_template(e['messages'], tokenize=False)})

In [ ]:
# 6. L'addestramento: solo le risposte insegnano
import time
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback
from unsloth.chat_templates import train_on_responses_only
passi = []
class Registro(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and 'loss' in logs:
            passi.append({'passo': state.global_step, 'errore': float(logs['loss'])})
inizio = time.time()
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=dati,
    args=SFTConfig(dataset_text_field='text', max_seq_length=lunghezza, num_train_epochs=2, learning_rate=2e-4,
                   per_device_train_batch_size=2, gradient_accumulation_steps=4, logging_steps=1,
                   output_dir='uscita', report_to='none'),
    callbacks=[Registro()])
trainer = train_on_responses_only(trainer, instruction_part='<|im_start|>user\n', response_part='<|im_start|>assistant\n')
trainer.train()

In [ ]:
# 7. rapporto.json: l'errore a ogni passo (Prisma ne disegna la curva)
rapporto = {'versione': 1, 'passi': passi, 'esempi': len(dati), 'modelloDiPartenza': manifesto['modelloDiPartenza'],
            'minuti': round((time.time() - inizio) / 60), 'nomeModello': manifesto['nomeModello']}
json.dump(rapporto, open('rapporto.json', 'w', encoding='utf-8'), ensure_ascii=False)

In [ ]:
# 8. Il modello in GGUF q4_k_m e lo zip da scaricare
model.save_pretrained_gguf('vera-su-misura', tokenizer, quantization_method='q4_k_m')
gguf = [os.path.join('vera-su-misura', n) for n in os.listdir('vera-su-misura') if n.endswith('.gguf')]
with zipfile.ZipFile('vera-su-misura.zip', 'w', zipfile.ZIP_STORED) as z:
    for g in gguf:
        z.write(g, os.path.basename(g))
    z.write('rapporto.json')
print('Fatto: scarica vera-su-misura.zip e importalo in Prisma (Vera › Vera su misura).')